# IHP SG13G2 Octagon L2n0 (2 nH) — 30 GHz validation

This notebook validates the IHP **SG13_Octagon_L2n0** benchmark at **30 GHz**.

* Layout: the IHP `/testcase/SG13_Octagon_L2n0` simplified GDS (vendored under
  `nbs/data/ihp_validation/source`, Apache-2.0, pinned to IHP-Open-PDK
  `9439837`).
* Reference: THRU-de-embedded measured GSGSG S-parameters on dies `x2y7` and
  `x5y8`, with the reference plane at the TopMetal1 feed ends.
* Simulation: gsim Palace with the **authoritative IHP PDK stack**, TopMetal2
  winding, TopMetal1 feeds/underpass, a SUBGND reference plane, 5 µm refined
  mesh cells, 70 µm maximum cells, and two interlayer ports from SUBGND to
  TopMetal1 — the same topology and port definition used by IHP's
  `run_inductor_2port.py` / `palace_L2n0.py` examples.

The IHP OpenEMS script is the reference model when an OpenEMS runtime is
available. This notebook uses the IHP-provided measured S-parameters as the
reference and records the OpenEMS script settings it mirrors. The recorded
Palace run is committed so the notebook renders without rerunning the solver.

> Model note: this is a single-frequency validation at 30 GHz with a coarse
> 5 µm mesh, matching the published IHP OpenEMS mesh setting. The measured SRF
> is ~25.5 GHz and the IHP OpenEMS SRF is ~22.7 GHz, so 30 GHz is above the
> resonance and the extracted series impedance is dominated by the
> de-embedded fixture/parasitics. The S-parameter comparison is the robust
> quantity here; L/Q is reported as an effective series extraction.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import skrf as rf

from gsim.common.benchmarks import ihp_l2n0 as _l2n0

plt.rcParams["figure.dpi"] = 120
REPO = _l2n0.REPO
RUN = _l2n0.RUN_ROOT / "l2n0_30ghz"
print("recorded run:", RUN)
print("measured dies:", sorted(_l2n0.MEASURED))

In [ ]:
run = _l2n0.load_recorded_run(RUN)
sp = run.sparams
freq_ghz = np.asarray(sp.freq, dtype=float)
print("Palace ports:", sp.port_names, "| frequencies (GHz):", freq_ghz)
print("S matrix at", freq_ghz[0], "GHz:")
print(_l2n0.s_matrix(sp, 0))

In [ ]:
def measured_network(label: str):
    """Measured 2-port network for one die."""
    return rf.Network(str(_l2n0.MEASURED[label]))


def lq_from_s(s_matrix: np.ndarray, f_hz: float, z0: float = _l2n0.Z0_OHM):
    """Series L/Q and Z from a 2x2 S matrix."""
    z = 2.0 * z0 * (1.0 - s_matrix[1, 0]) / s_matrix[1, 0]
    inductance_nh = z.imag / (2.0 * np.pi * f_hz) * 1e9
    q = z.imag / z.real if z.real != 0 else float("nan")
    return inductance_nh, q, z


def measured_at(label: str, f_hz: float = _l2n0.TARGET_FREQ_HZ):
    network = measured_network(label)
    point = network.interpolate(rf.Frequency(f_hz / 1e9, f_hz / 1e9, 1, unit="GHz"))
    return point.s[0], lq_from_s(point.s[0], f_hz)


rows = []
sim_s = _l2n0.s_matrix(sp, 0)
sim_l, sim_q, sim_z = _l2n0.lq_at(sp, 0)
rows.append(
    (
        "Palace (gsim)",
        20 * np.log10(abs(sim_s[0, 0])),
        20 * np.log10(abs(sim_s[1, 0])),
        sim_l,
        sim_q,
    )
)
for label in ("die_x2y7", "die_x5y8"):
    s, (l, q, z) = measured_at(label)
    rows.append((label, 20 * np.log10(abs(s[0, 0])), 20 * np.log10(abs(s[1, 0])), l, q))
print(f"{'case':18} {'|S11| dB':>9} {'|S21| dB':>9} {'L (nH)':>8} {'Q':>8}")
for name, s11, s21, l, q in rows:
    print(f"{name:18} {s11:9.2f} {s21:9.2f} {l:8.3f} {q:8.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

# Measured frequency sweeps
for label, color in (("die_x2y7", "C0"), ("die_x5y8", "C1")):
    n = measured_network(label)
    axes[0].plot(
        n.f / 1e9,
        20 * np.log10(np.abs(n.s[:, 0, 0])),
        color=color,
        lw=1.0,
        label=f"{label} S11",
    )
    axes[0].plot(
        n.f / 1e9,
        20 * np.log10(np.abs(n.s[:, 1, 0])),
        color=color,
        lw=1.0,
        ls="--",
        label=f"{label} S21",
    )
    # L/Q across the measured band (documented effective extraction)
    z0_meas = float(np.real(np.asarray(n.z0).reshape(-1)[0]))
    z = 2.0 * z0_meas * (1.0 - n.s[:, 1, 0]) / n.s[:, 1, 0]
    f = n.f
    axes[1].plot(
        f / 1e9, z.imag / (2 * np.pi * f) * 1e9, color=color, lw=1.0, label=f"{label} L"
    )
axes[0].axvline(30, color="k", lw=0.8, ls=":")
axes[1].axvline(30, color="k", lw=0.8, ls=":")

# Palace at 30 GHz
axes[0].plot(
    [30], [20 * np.log10(abs(sim_s[0, 0]))], "o", ms=7, color="C3", label="Palace S11"
)
axes[0].plot(
    [30], [20 * np.log10(abs(sim_s[1, 0]))], "s", ms=7, color="C3", label="Palace S21"
)
axes[1].plot([30], [sim_l], "o", ms=7, color="C3", label="Palace L")

axes[0].set(
    title="Measured and Palace S-parameters", xlabel="frequency (GHz)", ylabel="dB"
)
axes[0].set_ylim(-45, 5)
axes[0].legend(fontsize=7, ncol=2)
axes[0].grid(alpha=0.3)
axes[1].set(
    title="Effective series inductance", xlabel="frequency (GHz)", ylabel="L (nH)"
)
axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Comparison table at the target frequency
markdown = [
    "| case | |S11| (dB) | |S21| (dB) | effective L (nH) | effective Q |",
    "| --- | --- | --- | --- | --- |",
]
for name, s11, s21, l, q in rows:
    markdown.append(f"| {name} | {s11:.2f} | {s21:.2f} | {l:.3f} | {q:.2f} |")
print("\n".join(markdown))

## What the IHP OpenEMS script specifies

The IHP openEMS example (`run_inductor_2port.py`) uses:

| setting | IHP openEMS | this notebook |
| --- | --- | --- |
| layout | `L_2n0_twoport.gds` / `L_2n0_simplified.gds` | vendored `L_2n0_simplified.gds` |
| ports | via ports SUBGND → TopMetal1, 50 Ω | interlayer ports SUBGND → TopMetal1, 50 Ω |
| frequency | 0–30 GHz, 401 points | single 30 GHz validation point |
| mesh | 1 µm refined in conductor, 200 µm margin | 5 µm refined, 70 µm max, 150 µm margin (published coarse Palace example) |
| boundary | PEC | Palace default absorbing/airbox |

The measured S-parameters are the authoritative reference in this notebook.
The IHP guide reports measured SRF ≈ 25.5 GHz / peak Q ≈ 23.5 at 9 GHz versus
OpenEMS SRF ≈ 22.7 GHz. The two measured dies show the die-to-die spread.

## Limitations

* Single-frequency Palace run; the measured sweeps are shown for context.
* Coarse 5 µm mesh; conductor surfaces and the conformal passivation are not
  resolved as finely as the published IHP OpenEMS model.
* The extracted effective series L/Q at 30 GHz is above the measured SRF and is
  sensitive to the de-embedding/ground reference; S-parameters are the primary
  comparison.
* A full OpenEMS run would require the IHP openEMS runtime; the script settings
  above document the equivalent reference model, and the measured s2p files are
  the pinned fallback allowed by the validation plan.